In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import xgboost as xgb
import joblib
import os
from sklearn.metrics import accuracy_score

# ── All 10 tickers ─────────────────────────────────────────────────────────
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", 
           "NVDA", "V", "JPM", "ORCL", "COST"]

# ── Feature engineering function ───────────────────────────────────────────
def build_features(ticker):
    df = yf.download(ticker, start="2019-01-01", end="2024-12-31", progress=False)
    df.columns = df.columns.get_level_values(0)
    
    df["rsi"]         = ta.momentum.RSIIndicator(df["Close"]).rsi()
    df["macd"]        = ta.trend.MACD(df["Close"]).macd()
    df["macd_sig"]    = ta.trend.MACD(df["Close"]).macd_signal()
    df["bb_high"]     = ta.volatility.BollingerBands(df["Close"]).bollinger_hband()
    df["bb_low"]      = ta.volatility.BollingerBands(df["Close"]).bollinger_lband()
    df["vol_ma"]      = df["Volume"].rolling(20).mean()
    df["returns_1d"]  = df["Close"].pct_change(1)
    df["returns_5d"]  = df["Close"].pct_change(5)
    df["returns_20d"] = df["Close"].pct_change(20)
    df["volatility"]  = df["returns_1d"].rolling(20).std()
    df["ma_20"]       = df["Close"].rolling(20).mean()
    df["ma_50"]       = df["Close"].rolling(50).mean()
    df["ma_cross"]    = (df["ma_20"] > df["ma_50"]).astype(int)
    df["target"]      = (df["Close"].shift(-5) > df["Close"]).astype(int)
    df.dropna(inplace=True)
    return df

# ── Download and combine all tickers ───────────────────────────────────────
print("Downloading data for all 10 tickers...")
all_data = []

for ticker in TICKERS:
    print(f"  Fetching {ticker}...")
    df = build_features(ticker)
    df["ticker"] = ticker
    all_data.append(df)
    print(f"  {ticker}: {len(df)} rows")

combined = pd.concat(all_data)
print(f"\nTotal combined dataset: {len(combined)} rows")

# ── Features and target ────────────────────────────────────────────────────
features = ["rsi","macd","macd_sig","bb_high","bb_low",
            "vol_ma","returns_1d","returns_5d","returns_20d",
            "volatility","ma_20","ma_50","ma_cross"]

X = combined[features]
y = combined["target"]

# ── Chronological split ────────────────────────────────────────────────────
# Split each ticker individually to avoid future leakage
train_frames = []
test_frames  = []

for ticker in TICKERS:
    ticker_df = combined[combined["ticker"] == ticker]
    split = int(len(ticker_df) * 0.8)
    train_frames.append(ticker_df.iloc[:split])
    test_frames.append(ticker_df.iloc[split:])

train = pd.concat(train_frames)
test  = pd.concat(test_frames)

X_train = train[features]
y_train = train["target"]
X_test  = test[features]
y_test  = test["target"]

print(f"Training rows : {len(X_train)}")
print(f"Test rows     : {len(X_test)}")

# ── Train model ────────────────────────────────────────────────────────────
print("\nTraining XGBoost on all 10 stocks...")
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# ── Evaluate ───────────────────────────────────────────────────────────────
preds = model.predict(X_test)
overall_acc = accuracy_score(y_test, preds)

# Buy signal accuracy
buy_mask = preds == 1
buy_acc  = accuracy_score(y_test[buy_mask], preds[buy_mask]) if buy_mask.sum() > 0 else 0

print(f"\nOverall accuracy  : {overall_acc:.1%}")
print(f"Buy signal accuracy: {buy_acc:.1%}")

# ── Per ticker accuracy ────────────────────────────────────────────────────
print("\nPer-ticker accuracy on test set:")
print("-" * 35)
for ticker in TICKERS:
    mask  = test["ticker"] == ticker
    t_acc = accuracy_score(y_test[mask], preds[mask])
    buys  = preds[mask] == 1
    b_acc = accuracy_score(y_test[mask][buys], preds[mask][buys]) if buys.sum() > 0 else 0
    print(f"  {ticker:<6} Overall: {t_acc:.1%}  Buy acc: {b_acc:.1%}")

# ── Save model ─────────────────────────────────────────────────────────────
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/aapl_xgb_model.joblib")
print("\nMulti-stock model saved to models/aapl_xgb_model.joblib")
print("Ready to push to GitHub!")

# Check full results
print(f"Overall accuracy  : {overall_acc:.1%}")
print(f"Buy signal accuracy: {buy_acc:.1%}")
print("\nPer-ticker accuracy on test set:")
print("-" * 35)
for ticker in TICKERS:
    mask  = test["ticker"] == ticker
    t_acc = accuracy_score(y_test[mask], preds[mask])
    buys  = preds[mask] == 1
    b_acc = accuracy_score(y_test[mask][buys], preds[mask][buys]) if buys.sum() > 0 else 0
    print(f"  {ticker:<6} Overall: {t_acc:.1%}  Buy acc: {b_acc:.1%}")

for ticker in ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA"]:
    mask  = test["ticker"] == ticker
    t_acc = accuracy_score(y_test[mask], preds[mask])
    buys  = preds[mask] == 1
    b_acc = accuracy_score(y_test[mask][buys], preds[mask][buys]) if buys.sum() > 0 else 0
    print(f"  {ticker:<6} Overall: {t_acc:.1%}  Buy acc: {b_acc:.1%}")

  Fetching AAPL...
  AAPL: 1460 rows
  Fetching MSFT...
  MSFT: 1460 rows
  Fetching GOOGL...
  GOOGL: 1460 rows
  Fetching AMZN...
  AMZN: 1460 rows
  Fetching META...
  META: 1460 rows
  Fetching NVDA...
  NVDA: 1460 rows
  Fetching V...
  V: 1460 rows
  Fetching JPM...
  JPM: 1460 rows
  Fetching ORCL...
  ORCL: 1460 rows
  Fetching COST...
  COST: 1460 rows

Total combined dataset: 14600 rows
Training rows : 11680
Test rows     : 2920

Training XGBoost on all 10 stocks...

Overall accuracy  : 55.7%
Buy signal accuracy: 61.7%

Per-ticker accuracy on test set:
-----------------------------------
  AAPL   Overall: 50.3%  Buy acc: 56.9%
  MSFT   Overall: 56.8%  Buy acc: 58.8%
  GOOGL  Overall: 62.7%  Buy acc: 61.2%
  AMZN   Overall: 59.2%  Buy acc: 61.8%
  META   Overall: 51.0%  Buy acc: 72.8%
  NVDA   Overall: 47.9%  Buy acc: 56.1%
  V      Overall: 54.1%  Buy acc: 59.5%
  JPM    Overall: 69.2%  Buy acc: 71.7%
  ORCL   Overall: 54.5%  Buy acc: 54.9%
  COST   Overall: 50.7%  Buy acc: 7